In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow import keras
from keras.datasets import fashion_mnist
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)

## 1) Förbereda datan

- ladda in datan
- undersöka vilka klasser som finns
- se hur många bilder som finns
- identifiera eventuella obalanser


In [ ]:
#Ladda in datan
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images_full, train_labels_full), (test_images, test_labels) = fashion_mnist.load_data()


print("Bilder i train set:", train_images_full.shape)
print("Bilder i test set:", test_images.shape)
print("Bilder totalt:", len(train_labels_full) + len(test_labels))
print("Klasser:", np.unique(test_labels))

### Summering av dataset

- Datasetet innehåller 70,000 bilder där varje bild representeras i 28 x 28 pixels var.
- Datasetet är redan indelat i train-set - 60,000 bilder och test-set - 10,000. 
- Det finns 10 klasskategorier eller *labels* som bilderna kan hamna inom - *labels* är en vektor av integers från 0 till 9. Dessa representerar vilken sorts klädkategori bilder hamnar inom:

| Label       | Class       |
| ----------- | ----------- |
| 0 	      | T-shirt/top |
| 1 	      |  Trouser    |
| 2 	      |  Pullover   |
| 3 	      |  Dress      |
| 4 	      |  Coat       |
| 5 	      |  Sandal     |
| 6 	      |  Shirt      |
| 7 	      |  Sneaker    |
| 8 	      |  Bag        |
| 9 	      |  Ankle boot |

In [ ]:
# Identifiera eventuella obalanser

classes_train, counts_train = np.unique(train_labels_full, return_counts=True)
classes_test, counts_test = np.unique(test_labels, return_counts=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(classes_train, counts_train, color = 'blue')
axes[0].set_title('Klassfördelning i train set')
axes[0].set_xlabel('Klass')
axes[0].set_ylabel('Antal')

axes[1].bar(classes_test, counts_test, color = 'green')
axes[1].set_title('Klassfördelning i test set')
axes[1].set_xlabel('Klass')
axes[1].set_ylabel('Antal')

plt.tight_layout()
plt.show()


### Summering av klassfördelning

Klassfördelningen visar ett perfekt balanserat dataset igenom både train och test setet. 

In [ ]:
# Bild före preprocessing

plt.figure()
plt.imshow(train_images_full[0])
plt.colorbar()
plt.grid(False)
plt.show()

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Testa att normaliseringen funkade
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images_full[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels_full[i]])
plt.show()


## 2) Förbereda datan

- forma datan så att den fungerar med modellen:
    - skapa ett subset av träningsdatan
    - dela datan i train - val 
    - normalisera/skala bilderna


In [ ]:
def create_balanced_subset(X, y, samples_per_class=700, random_state=42):

    rng = np.random.default_rng(random_state)
    selected_indices = []

    for class_id in np.unique(y):
        class_indices = np.where(y == class_id)[0]

        if len(class_indices) < samples_per_class:
            raise ValueError(
                f"Klassen {class_id} har bara {len(class_indices)} exempel, "
                f"men samples_per_class={samples_per_class}"
            )
        
        chosen_indices = rng.choice(
            class_indices,
            size=samples_per_class,
            replace=False
        )

        selected_indices.extend(chosen_indices)
    
    selected_indices = np.array(selected_indices)
    rng.shuffle(selected_indices)

    return X[selected_indices], y[selected_indices]


X_subset_raw, y_subset = create_balanced_subset(
    train_images_full,
    train_labels_full,
    samples_per_class=700,
    random_state=42
)

print("X_subset_raw:", X_subset_raw.shape)
print("y_subset:", y_subset.shape)



In [ ]:
train_images, val_images, train_labels, val_labels = train_test_split(
    X_subset_raw,
    y_subset,
    test_size=0.20,
    random_state=42,
    stratify=y_subset
)

print("Train images:", train_images.shape)
print("Validation images:", val_images.shape)
print("Train labels:", train_labels.shape)
print("Validation labels:", val_labels.shape)

In [ ]:
train_images_norm = train_images.astype("float32") / 255.0
val_images_norm = val_images.astype("float32") / 255.0
test_images_norm = test_images.astype("float32") / 255.0

print("Train imgs min/max:", train_images_norm.min(), train_images_norm.max())
print("Validation imgs min/max:", val_images_norm.min(), val_images_norm.max())
print("Test imgs shape:", test_images_norm.shape)

In [ ]:
train_images_norm = train_images_norm[..., np.newaxis]
val_images_norm = val_images_norm[..., np.newaxis]
test_images_norm = test_images_norm[..., np.newaxis]

print("Train imgs shape efter kanal-dimension:", train_images_norm.shape)
print("Validation imgs shape efter kanal-dimension:", val_images_norm.shape)
print("Test imgs shape efter kanal-dimension:", test_images_norm.shape)

## 3) Bygga en modell

- skapa en första fungerande modell
- använda neurala nätverk (t.ex. CNN)
- säkerställa att modellen kan tränas


In [ ]:
# Bygger en baseline CNN-model

def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),       # 128 neuroner 
        layers.Dense(10, activation="softmax")      # 10 klassser
    ])
    
    model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
    )
    
    return model

In [ ]:
baseline_model = build_baseline_model()
baseline_model.summary()

Vi skapade en baseline-modell med CNN-arkitektur. Modellen består av två convolutional layers (Conv2D) som används för att identifiera mönster i bilderna, följt av max pooling som minskar bilddimensionerna. Därefter används ett flatten-lager och dense-lager för klassificering. Det sista lagret har 10 neuroner eftersom Fashion MNIST innehåller 10 klasser. Softmax används för att modellen ska ge sannolikheter för varje klass.

In [ ]:
history_baseline = baseline_model.fit(
    train_images_norm,
    train_labels,
    validation_data = (val_images_norm, val_labels),
    epochs=5,
    batch_size=32
)

### Träna baseline-modellen

Efter att baseline-modellen har byggts och compile:ats tränas den på vårt normaliserade träningsdata. 
Vi använder validation data för att kunna följa hur modellen presterar på data som den inte tränas direkt på.

I den första baseline använder vi 5 epochs och batch size 32. 
Resultatet sparas i `history_baseline`, vilket gör att vi senare kan visualisera utvecklingen av loss och accuracy under träningen.

In [ ]:
# Funktion för att plotta träningskurvor för loss och accuracy.

def plot_history(history, title="Träningskurvor"):
    history_df = pd.DataFrame(history.history)

    epochs = range(1, len(history_df) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history_df["loss"], label="Traning loss")
    plt.plot(epochs, history_df["val_loss"], label="Validation loss")
    plt.grid(True)
    plt.xlabel("Epok")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history_df["accuracy"], label="Traning accuracy")
    plt.plot(epochs, history_df["val_accuracy"], label="Validation accuracy")
    plt.grid(True)
    plt.xlabel("Epok")
    plt.ylabel("Accuracy")
    plt.title("Accuracy")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_history(history_baseline, title="Baseline-CNN på subset")

Graferna visar att modellen lär sig stabilt då loss minskar och accuracy ökar över epokerna.   
Validation-kurvorna följer training-kurvorna ganska nära, vilket tyder på att modellen fungerar bra utan tydlig overfitting.   
Resultaten är dock baserade på ett subset av datan.

In [ ]:
def predict_classes(model, X):
    y_proba = model.predict(X, verbose=0)
    y_pred = np.argmax(y_proba, axis=1)

    return y_pred, y_proba

In [ ]:

baseline_val_pred, baseline_val_proba = predict_classes(
    baseline_model,
    val_images_norm
)

baseline_val_accuracy = accuracy_score(val_labels, baseline_val_pred)

print(f"Baseline validation accuracy: {baseline_val_accuracy:.4f}")

In [ ]:
# Plot för confusionmatrix

def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion matrix"):

    labels = np.arange(len(class_names))

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    display.plot(
        ax=ax,
        xticks_rotation=45,
        values_format="d",
        cmap="Oranges",
        colorbar=False
    )

    ax.set_xlabel("Predicted label", fontsize=14)
    ax.set_ylabel("True label", fontsize=14)

    plt.title(title)
    plt.tight_layout()
    plt.show()

    return cm

In [ ]:
baseline_cm = plot_confusion_matrix(
    val_labels,
    baseline_val_pred,
    class_names,
    title="Baseline-CNN - confusion matrix på validation-data"
)

Confusion matrixen visar att modellen klassificerar flera klasser väldigt bra, särskilt Trouser, Sandal och Ankle boot där många prediktioner hamnar rätt.   
Modellen verkar däremot ha svårare att skilja mellan vissa liknande klädesplagg, exempelvis Shirt, T-shirt/top, Coat och Pullover, där fler felklassificeringar syns.